# 🔬 JORINOVA NEXUS — Cervical cytology (Pap) CLASSIFIER (SIPaKMeD)

Classifies a cervical cell into the 5 **SIPaKMeD** classes (dyskeratotic, koilocytotic,
metaplastic, parabasal, superficial-intermediate). Classification model (`yolov8m-cls`);
metric = **top-1 accuracy**, target ~0.95.

Produces `cytology.pt`; the predicted class resolves to its Bethesda finding via
`backend/ai_services/cytology_findings.json`.

> **Data source: KAGGLE** — `marinaeplissiti/sipakmed` (4049 real cell crops). Upload `kaggle.json` in step 4.

**Before you start:** Runtime -> **Change runtime type** -> **T4 GPU**.

## 1. Check the GPU

In [ ]:
!nvidia-smi -L || echo 'NO GPU — set Runtime > Change runtime type > T4 GPU, then rerun'

## 2. Install training dependencies

In [ ]:
%pip -q install ultralytics kaggle
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 3. Mount Drive (checkpoints survive a runtime reset)

In [ ]:
import os
try:
    from google.colab import drive; drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/nexus_cytology', exist_ok=True)
    print('Drive ready: /content/drive/MyDrive/nexus_cytology')
except Exception as e:
    print('Drive not mounted (optional):', e)

## 4. Get the training data (KAGGLE — SIPaKMeD)
Upload your `kaggle.json` (kaggle.com → Settings → API → **Create New Token**). The cell downloads
**`marinaeplissiti/sipakmed`** (5 Pap-cell classes, 4049 crops), reads the class from each folder
path, and renames it to the `cytology_findings.json` keys.

In [ ]:
# ── 4. Cytology data: KAGGLE SIPaKMeD (5-class Pap) -> train/val class folders ──
# marinaeplissiti/sipakmed: 4049 cell crops, 5 classes. Class is read from the folder PATH
# (handles nested im_<Class>/CROPPED/ layouts) and renamed to cytology_findings.json keys.
import os, glob, shutil, subprocess, sys, random
if not os.path.exists('/root/.kaggle/kaggle.json'):
    from google.colab import files
    print('Upload kaggle.json (kaggle.com -> Settings -> API -> Create New Token) ...'); up = files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    name = 'kaggle.json' if os.path.exists('kaggle.json') else list(up)[0]
    shutil.move(name, '/root/.kaggle/kaggle.json'); os.chmod('/root/.kaggle/kaggle.json', 0o600)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'], check=True)

RAW = '/content/data/cyto_raw'
if os.path.exists(RAW): shutil.rmtree(RAW)
os.makedirs(RAW, exist_ok=True)
rc = subprocess.run(['kaggle', 'datasets', 'download', '-d', 'marinaeplissiti/sipakmed', '-p', RAW, '--unzip']).returncode
assert rc == 0, 'Kaggle download failed — accept the dataset rules on kaggle.com + check kaggle.json.'

def canon(n):
    n = n.lower()
    if 'dyskeratotic' in n: return 'dyskeratotic'
    if 'koilocyto' in n: return 'koilocyte'
    if 'metaplastic' in n: return 'metaplastic'
    if 'parabasal' in n: return 'parabasal'
    if 'superficial' in n or 'intermediate' in n: return 'superficial_intermediate'
    return None
def cls_of(path):
    for part in reversed(path.replace('\\', '/').split('/')):
        k = canon(part)
        if k: return k
    return None

exts = ('*.bmp', '*.jpg', '*.jpeg', '*.png')
imgs = [p for e in exts for p in glob.glob(RAW + '/**/' + e, recursive=True)]
assert imgs, 'No images under ' + RAW
crop = [p for p in imgs if 'crop' in p.lower()]
if crop: imgs = crop                                  # prefer single-cell CROPPED images
CLS = '/content/data/cyto_cls'
if os.path.exists(CLS): shutil.rmtree(CLS)
buckets = {}
for p in imgs:
    k = cls_of(p)
    if k: buckets.setdefault(k, []).append(p)
assert len(buckets) >= 2, 'Fewer than 2 SIPaKMeD classes recognised — print imgs[:5] and send me the paths.'
for k, ps in buckets.items():
    random.shuffle(ps); nval = max(1, len(ps) // 6)
    for i, p in enumerate(ps):
        split = 'val' if i < nval else 'train'
        out = os.path.join(CLS, split, k); os.makedirs(out, exist_ok=True)
        shutil.copy(p, os.path.join(out, '%s_%d%s' % (k, i, os.path.splitext(p)[1])))
DATA_DIR = CLS
counts = {s: {c: len(os.listdir(os.path.join(CLS, s, c))) for c in sorted(os.listdir(os.path.join(CLS, s)))}
          for s in ('train', 'val') if os.path.isdir(os.path.join(CLS, s))}
print('DATA_DIR =', DATA_DIR); print('classes:', sorted(buckets)); print('counts:', counts)

## 5. Choose base weights (fine-tune, never from scratch)

In [ ]:
BASE_WEIGHTS = 'yolov8m-cls.pt'   # classification model (yolov8s-cls = faster/less accurate)
print('fine-tuning (classification) from:', BASE_WEIGHTS)

## 6. Fine-tune the classifier
Metric is **top-1 accuracy** (not mAP). SIPaKMeD cell crops → `imgsz=320`, `yolov8m-cls`,
~120 epochs, cosine LR + light dropout/erasing to curb overfit. Microscopy has no canonical
orientation, so flips + rotation are safe augmentation.

In [ ]:
from ultralytics import YOLO
import os
PROJECT = '/content/drive/MyDrive/nexus_cytology/runs'
try: os.makedirs(PROJECT, exist_ok=True)
except Exception: PROJECT = '/content/runs/classify'
model = YOLO(BASE_WEIGHTS)   # yolov8m-cls -> classification (SIPaKMeD is class-labelled)
model.train(data=DATA_DIR, epochs=120, imgsz=320, batch=64,
            project=PROJECT, name='cytology', pretrained=True, patience=30, plots=True,
            cos_lr=True, dropout=0.2, fliplr=0.5, flipud=0.5, degrees=180,
            hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, erasing=0.2)
m = model.val(); print(f'top1 = {m.top1:.3f}   top5 = {m.top5:.3f}')

## 6b. RESUME if a run disconnects (run INSTEAD of step 6)

In [ ]:
import os, glob
from ultralytics import YOLO
runs = sorted(glob.glob('/content/drive/MyDrive/nexus_cytology/runs/**/weights/last.pt', recursive=True), key=os.path.getmtime)
loose = [p for p in sorted(glob.glob('/content/drive/MyDrive/nexus_cytology/**/*.pt', recursive=True)
                           + glob.glob('/content/*.pt'), key=os.path.getmtime)
         if 'yolov8' not in os.path.basename(p).lower()]
assert runs or loose, 'No checkpoint yet — run step 6 first.'
ckpt = (runs or loose)[-1]; print('checkpoint:', ckpt)
model = None
try:
    if ckpt in runs: model = YOLO(ckpt); model.train(resume=True)
    else: raise RuntimeError('loose')
except Exception as e:
    print('continue-from-weights:', str(e)[:80]); model = None
if model is None:
    assert 'DATA_DIR' in globals(), 'Re-run step 4 first so DATA_DIR is set.'
    model = YOLO(ckpt); model.train(data=DATA_DIR, epochs=120, imgsz=320, batch=64,
        project='/content/drive/MyDrive/nexus_cytology/runs', name='cytology_cont', pretrained=True, patience=30,
        cos_lr=True, dropout=0.2, fliplr=0.5, flipud=0.5, degrees=180, erasing=0.2)
m = model.val(); print(f'top1 = {m.top1:.3f}   top5 = {m.top5:.3f}')

## 7. Export the model (Drive backup + download)

In [ ]:
import shutil, glob, os
DRIVE = '/content/drive/MyDrive/nexus_cytology'
best = sorted([c for c in glob.glob('/content/**/weights/best.pt', recursive=True)
               + glob.glob(DRIVE + '/**/weights/best.pt', recursive=True) if os.path.isfile(c)],
              key=os.path.getmtime)[-1]
print('using weights:', best)
from ultralytics import YOLO; print('classes:', YOLO(best).model.names)
try: os.makedirs(DRIVE, exist_ok=True); shutil.copy(best, DRIVE + '/cytology.pt'); print('Drive ->', DRIVE + '/cytology.pt')
except Exception as e: print('drive copy skipped:', e)
from google.colab import files; files.download(best)

## 8. Put it in the app
1. Rename the download to **`cytology.pt`** at **`backend/models/cytology/cytology.pt`**, commit, push.
2. Auto-loads for `image_type == "cytology"`; the predicted class resolves via `backend/ai_services/cytology_findings.json`. The vision service **detects classification models automatically** (reads top-1 + top-k, no code change).
3. On Render, build with `INSTALL_ML=1` (>=2 GB); else Claude vision reads the field.

> ⚠️ Screening aid only — a cytopathologist reports. If your dataset's class names differ from the map keys, send me the printed `classes:` list and I'll add aka-aliases.